# Analiza danych parku rozrywki AiBtcQuantLandia
## 1. Wstęp

Raport przedstawia analizę danych zgromadzonych w relacyjnej bazie danych
parku rozrywki **AiBtcQuantLandia**.

Celem raportu jest:
- analiza popularności atrakcji,
- porównanie kosztów i przychodów działalności,
- ocena liczby odwiedzających na przestrzeni czasu.

**Wszystkie wyniki, wykresy oraz wnioski są generowane automatycznie**
na podstawie aktualnego stanu bazy danych.  
Zmiana danych wejściowych powoduje automatyczną aktualizację raportu.

## 2. Połączenie z bazą
Analiza opiera się na danych przechowywanych w relacyjnej bazie danych MySQL.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from budowa import Baza 

try:
    baza = Baza()
    print("Pomyślnie połączono z bazą AiBtcQuantLandia.")
except Exception as e:
    print(f"Błąd połączenia: {e}")

## 3. Analiza danych 
### a) Która atrakcja przynosi najwiekszy przychód?

In [ ]:
query = """
SELECT a.attraction_name, SUM(p.amount) AS przychod
FROM attractions a
JOIN prices pr ON a.attraction_id = pr.attraction_id
JOIN payment_ticket pt ON pr.ticket_id = pt.ticket_id
JOIN payments p ON pt.payment_id = p.payment_id
GROUP BY a.attraction_name
ORDER BY przychod DESC
"""
baza.cursor.execute(query)
rows = baza.cursor.fetchall()

attractions = [r[0] for r in rows]
revenues = [r[1] for r in rows]

import matplotlib.pyplot as plt

plt.bar(attractions, revenues, color='pink')
plt.xlabel("Atrakcja")
plt.ylabel("Przychód (PLN)")
plt.title("Przychody atrakcji w AiBtcQuantLandia")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

#print(f"Najwięcej zarabia atrakcja: .")

## 4. Analiza bezpieczeństwa atrakcji

In [ ]:
print("\n" + "=" * 70)
print("ANALIZA BEZPIECZEŃSTWA: Czy któreś atrakcje są zbyt niebezpieczne?")
print("=" * 70)

query_safety = """
SELECT 
    a.attraction_name,
    a.vr,
    COUNT(i.incident_id) as liczba_incydentow,
    AVG(it.risk_level) as srednie_ryzyko,
    GROUP_CONCAT(DISTINCT it.name SEPARATOR ', ') as typy_incydentow,
    SUM(p.amount) as przychod_atrakcji,
    COUNT(DISTINCT p.guest_id) as liczba_gosci
FROM attractions a
LEFT JOIN incidents i ON a.attraction_id = i.attraction_id
LEFT JOIN incident_type it ON i.incident_type_id = it.incident_type_id
LEFT JOIN prices pr ON a.attraction_id = pr.attraction_id
LEFT JOIN payment_ticket pt ON pr.ticket_id = pt.ticket_id
LEFT JOIN payments p ON pt.payment_id = p.payment_id
GROUP BY a.attraction_id, a.attraction_name, a.vr
ORDER BY liczba_incydentow DESC
"""

baza.cursor.execute(query_safety)
rows_safety = baza.cursor.fetchall()

if len(rows_safety) > 0:
    print("Analiza incydentów w atrakcjach:")
    print("-" * 100)
    print(f"{'Atrakcja':<30} {'VR':<5} {'Incydenty':<10} {'Śr. ryzyko':<12} {'Przychód (PLN)':<15} {'Goście':<10}")
    print("-" * 100)
    
    for row in rows_safety:
        attraction_name, vr, incidents, avg_risk, incident_types, revenue, guests = row
        vr_str = "TAK" if vr else "NIE"
        avg_risk_str = f"{avg_risk:.2f}" if avg_risk else "0.00"
        revenue_str = f"{revenue:,.0f}" if revenue else "0"
        guests_str = f"{guests}" if guests else "0"
        
        print(f"{attraction_name:<30} {vr_str:<5} {incidents:<10} {avg_risk_str:<12} {revenue_str:<15} {guests_str:<10}")
    
    total_incidents = sum(row[2] for row in rows_safety)
    attractions_with_incidents = sum(1 for row in rows_safety if row[2] > 0)
    total_attractions = len(rows_safety)
    
    print(f"\n📊 STATYSTYKI OGÓLNE:")
    print(f"   Łączna liczba atrakcji: {total_attractions}")
    print(f"   Atrakcje z incydentami: {attractions_with_incidents}")
    print(f"   Atrakcje bez incydentów: {total_attractions - attractions_with_incidents}")
    print(f"   Łączna liczba incydentów: {total_incidents}")
    
    dangerous = [row for row in rows_safety if row[2] > 0]
    
    if dangerous:
        print(f"\n⚠️ NAJBARDZIEJ NIEBEZPIECZNE ATRAKCJE (powyżej 2 incydentów):")
        for row in dangerous:
            if row[2] > 2:
                print(f"   • {row[0]}: {row[2]} incydentów (średnie ryzyko: {row[3]:.1f})")
                if row[4]:
                    print(f"     Typy incydentów: {row[4][:100]}...")
        
        dangerous_sorted = sorted([r for r in dangerous if r[3] is not None], key=lambda x: x[3], reverse=True)[:5]
        
        print(f"\n📈 ATRAKCJE Z NAJWYŻSZYM ŚREDNIM RYZYKIEM:")
        for row in dangerous_sorted:
            print(f"   • {row[0]}: średnie ryzyko {row[3]:.2f} ({row[2]} incydentów)")
    
    print(f"\n📊 ANALIZA KORELACJI PRZYCHÓD VS BEZPIECZEŃSTWO:")
    
    high_risk = [r for r in rows_safety if r[3] is not None and r[3] >= 4]
    medium_risk = [r for r in rows_safety if r[3] is not None and 2 <= r[3] < 4]
    low_risk = [r for r in rows_safety if r[3] is not None and r[3] < 2]
    no_incidents = [r for r in rows_safety if r[2] == 0]
    
    print(f"   Wysokie ryzyko (≥4): {len(high_risk)} atrakcji")
    print(f"   Średnie ryzyko (2-4): {len(medium_risk)} atrakcji")
    print(f"   Niskie ryzyko (<2): {len(low_risk)} atrakcji")
    print(f"   Bez incydentów: {len(no_incidents)} atrakcji")
    
    def avg_revenue(category):
        revenues = [r[5] or 0 for r in category]
        return sum(revenues) / len(revenues) if revenues else 0
    
    print(f"\n💰 ŚREDNI PRZYCHÓD WG KATEGORII RYZYKA:")
    print(f"   Wysokie ryzyko: {avg_revenue(high_risk):,.0f} PLN")
    print(f"   Średnie ryzyko: {avg_revenue(medium_risk):,.0f} PLN")
    print(f"   Niskie ryzyko: {avg_revenue(low_risk):,.0f} PLN")
    print(f"   Bez incydentów: {avg_revenue(no_incidents):,.0f} PLN")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    top_dangerous = sorted(dangerous, key=lambda x: x[2], reverse=True)[:10]
    if top_dangerous:
        ax1.bar([r[0][:15] + "..." for r in top_dangerous], [r[2] for r in top_dangerous], color=['red' if r[1] else 'orange' for r in top_dangerous])
        ax1.set_xlabel('Atrakcja')
        ax1.set_ylabel('Liczba incydentów')
        ax1.set_title('Top 10 atrakcji z najwięcej incydentami')
        ax1.set_xticklabels([r[0][:15] + "..." for r in top_dangerous], rotation=45, ha='right')
    
    revenues = [r[5] or 0 for r in rows_safety]
    incidents = [r[2] for r in rows_safety]
    
    scatter = ax2.scatter(incidents, revenues, c=[r[3] or 0 for r in rows_safety], cmap='viridis', alpha=0.6, s=100)
    ax2.set_xlabel('Liczba incydentów')
    ax2.set_ylabel('Przychód (PLN)')
    ax2.set_title('Korelacja: Przychód vs Incydenty')
    ax2.grid(True, alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label('Średni poziom ryzyka')
    
    plt.tight_layout()
    plt.savefig('wykres_bezpieczenstwo.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "=" * 70)
    print("WNIOSKI I REKOMENDACJE DLA ZARZĄDU")
    print("=" * 70)
    
    if dangerous:
        most_dangerous = max(dangerous, key=lambda x: x[2])
        highest_risk = max([r for r in dangerous if r[3] is not None], key=lambda x: x[3], default=None)
        
        print(f"\n1. NAJBARDZIEJ PROBLEMATYCZNE ATRAKCJE:")
        print(f"   • Pod względem liczby incydentów: {most_dangerous[0]}")
        print(f"     - {most_dangerous[2]} incydentów")
        if most_dangerous[3]:
            print(f"     - Średnie ryzyko: {most_dangerous[3]:.2f}")
        else:
            print(f"     - Brak danych o ryzyku")
        
        if highest_risk and highest_risk[0] != most_dangerous[0]:
            print(f"   • Pod względem poziomu ryzyka: {highest_risk[0]}")
            print(f"     - Średnie ryzyko: {highest_risk[3]:.2f}")
            print(f"     - {highest_risk[2]} incydentów")
    
    print(f"\n2. ANALIZA FINANSOWA BEZPIECZEŃSTWA:")
    
    high_risk_revenue = avg_revenue(high_risk)
    no_incidents_revenue = avg_revenue(no_incidents)
    
    if high_risk_revenue > 0 and no_incidents_revenue > 0:
        revenue_difference = high_risk_revenue - no_incidents_revenue
        if revenue_difference > 0:
            print(f"   • Atrakcje wysokiego ryzyka generują ŚREDNIO {revenue_difference:,.0f} PLN WIĘCEJ")
            print(f"     niż atrakcje bez incydentów.")
            print(f"     To może sugerować, że 'niebezpieczne' atrakcje są bardziej popularne.")
        else:
            print(f"   • Atrakcje wysokiego ryzyka generują ŚREDNIO {abs(revenue_difference):,.0f} PLN MNIEJ")
            print(f"     niż atrakcje bez incydentów.")
    
    print(f"\n3. REKOMENDACJE DLA POLIS UBEZPIECZENIOWYCH:")
    
    if high_risk:
        print(f"   • Rozważyć PODWYŻSZENIE SKŁADKI ubezpieczeniowej dla {len(high_risk)} atrakcji")
        print(f"     z wysokim poziomem ryzyka (≥4)")
        for att in high_risk[:3]:
            print(f"     - {att[0]}: ryzyko {att[3]:.2f}, {att[2]} incydentów")
    
    if no_incidents:
        print(f"\n   • Rozważyć OBNIŻENIE SKŁADKI dla {len(no_incidents)} atrakcji")
        print(f"     które NIE MIAŁY ŻADNYCH incydentów")
        safe_and_popular = sorted(no_incidents, key=lambda x: (x[5] or 0), reverse=True)[:3]
        if safe_and_popular:
            print(f"     (Polecane do promocji jako 'bezpieczne atrakcje'):")
            for att in safe_and_popular:
                print(f"     - {att[0]}: {att[5]:,.0f} PLN przychodu")
    
    print(f"\n4. REKOMENDACJE TECHNICZNE I ORGANIZACYJNE:")
    
    vr_dangerous = [r for r in dangerous if r[1]]
    real_dangerous = [r for r in dangerous if not r[1]]
    
    print(f"   • Atrakcje VR z incydentami: {len(vr_dangerous)}")
    print(f"   • Atrakcje rzeczywiste z incydentami: {len(real_dangerous)}")
    
    if len(real_dangerous) > len(vr_dangerous):
        print(f"     → Więcej incydentów w atrakcjach RZECZYWISTYCH.")
        print(f"     Rozważyć dodatkowe szkolenia dla operatorów.")
    else:
        print(f"     → Więcej incydentów w atrakcjach VR.")
        print(f"     Sprawdzić stan techniczny sprzętu VR.")
    
    print(f"\n5. PLAN DZIAŁANIA NA NAJBLIŻSZE 3 MIESIĄCE:")
    print(f"   a) Przegląd techniczny 3 najbardziej ryzykownych atrakcji")
    print(f"   b) Dodatkowe szkolenie BHP dla operatorów")
    print(f"   c) Analiza kosztów napraw vs nowe inwestycje")
    print(f"   d) Kampania marketingowa 'bezpieczne atrakcje'")
    
    print(f"\n✅ Analiza zakończona. Wykres zapisano jako 'wykres_bezpieczenstwo.png'")
    
else:
    print("Brak danych do analizy bezpieczeństwa.")